In [1]:
#!pip -q install ipywidgets opencv-python-headless

import cv2
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output
from google.colab import output
output.enable_custom_widget_manager()
plt.rcParams["figure.figsize"] = (10, 6)


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
numba 0.61.2 requires numpy<2.3,>=1.24, but you have numpy 2.4.2 which is incompatible.
tensorflow 2.19.0 requires numpy<2.2.0,>=1.26.0, but you have numpy 2.4.2 which is incompatible.

[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


ModuleNotFoundError: No module named 'google.colab'

In [ ]:
from google.colab import files

def load_image_bgr_from_upload():
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError("No file uploaded.")
    name = next(iter(uploaded.keys()))
    data = uploaded[name]
    arr = np.frombuffer(data, np.uint8)
    img_bgr = cv2.imdecode(arr, cv2.IMREAD_COLOR)
    if img_bgr is None:
        raise RuntimeError("Could not decode the uploaded file as an image.")
    return img_bgr, name

img_bgr, img_name = load_image_bgr_from_upload()
print("Loaded:", img_name, "shape:", img_bgr.shape)


Saving duckies.jpg to duckies.jpg
Loaded: duckies.jpg shape: (1500, 1500, 3)


In [ ]:
img_hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)

h_range = widgets.IntRangeSlider(value=[0,179], min=0, max=179, step=1, description='H')
s_range = widgets.IntRangeSlider(value=[0,255], min=0, max=255, step=1, description='S')
v_range = widgets.IntRangeSlider(value=[0,255], min=0, max=255, step=1, description='V')
show_mask = widgets.Checkbox(value=True, description="Show mask")

ui = widgets.VBox([h_range, s_range, v_range, show_mask])
out = widgets.Output()

def apply_threshold_and_show(_=None):
    h_lo, h_hi = h_range.value
    s_lo, s_hi = s_range.value
    v_lo, v_hi = v_range.value

    lower = np.array([h_lo, s_lo, v_lo], dtype=np.uint8)
    upper = np.array([h_hi, s_hi, v_hi], dtype=np.uint8)

    mask = cv2.inRange(img_hsv, lower, upper)
    filtered_bgr = cv2.bitwise_and(img_bgr, img_bgr, mask=mask)

    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    filtered_rgb = cv2.cvtColor(filtered_bgr, cv2.COLOR_BGR2RGB)

    with out:
        clear_output(wait=True)
        fig, axes = plt.subplots(1, 3, figsize=(18, 6))
        axes[0].imshow(img_rgb); axes[0].axis("off"); axes[0].set_title("Original")
        axes[1].imshow(mask, cmap="gray"); axes[1].axis("off"); axes[1].set_title("Mask")
        axes[2].imshow(filtered_rgb); axes[2].axis("off"); axes[2].set_title("Filtered")
        plt.show()

for w in [h_range, s_range, v_range, show_mask]:
    w.observe(apply_threshold_and_show, names="value")

display(ui, out)
apply_threshold_and_show()


Output()